# Core Data Types and Type Conversion

**Topic:** `1.5` &nbsp;|&nbsp; **Module 1: Getting Started with Python**

*Reference implementations for all 10 exercises plus the robotics challenge.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M1)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [01_getting_started/README.md](../README.md)

> **Compare, do not copy**
>
> Work each exercise yourself first. When you open this notebook, compare your approach with the reference: check the reasoning, the edge cases and the complexity. If your solution differs, that is fine — verify it with the tests provided and keep whichever is clearer.

## Solution 1 — Name the type of a value

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** type(), builtins

**Explanation and reasoning**

type() returns the class object and __name__ reads its name, so the answer is derived from the value rather than from a mapping the author maintained. That means custom classes and future built-ins are reported correctly without any extra code.

In [ ]:
def type_name(value) -> str:
    return type(value).__name__

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

NoneType is reported for None because its class is named NoneType.

**Alternative approaches**

type(value).__qualname__ distinguishes nested classes.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert type_name(48.0) == 'float'
assert type_name(None) == 'NoneType'
assert type_name([]) == 'list'

---

## Solution 2 — Convert external text to a float safely

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** float(), exceptions, defaults

**Explanation and reasoning**

The bool guard comes first precisely because bool is a subclass of int, so without it True would silently become 1.0. Passing real numbers through avoids a pointless string round trip, and stripping handles the whitespace that serial output usually carries.

In [ ]:
def to_float(raw, default: float = 0.0) -> float:
    if isinstance(raw, bool):
        return default
    if isinstance(raw, (int, float)):
        return float(raw)
    try:
        return float(str(raw).strip())
    except (TypeError, ValueError):
        return default

**Complexity**

Time O(n) in the text length; space O(n).

**Edge cases and failure modes**

'  12  ' converts to 12.0; 'n/a' and None both return the default.

**Alternative approaches**

A single try/except around float(raw) is shorter but mishandles booleans.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert to_float('87.5') == 87.5
assert to_float('  12  ') == 12.0
assert to_float(True) == 0.0
assert to_float('n/a', default=-1.0) == -1.0

---

## Solution 3 — Clamp a value into a range

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** comparisons, conditionals, types

**Explanation and reasoning**

Validating the bounds before clamping means a reversed range is reported as the programming error it is, rather than silently returning a value that satisfies neither constraint. Returning the bound objects themselves preserves the numeric type of the caller.

In [ ]:
def clamp(value, low, high):
    if low > high:
        raise ValueError(f'low {low} is above high {high}')
    if value < low:
        return low
    if value > high:
        return high
    return value

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

low equal to high returns that value for everything in range.

**Alternative approaches**

max(low, min(high, value)) is shorter but hides the reversed-bounds bug.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert clamp(118.0, 0, 100) == 100
assert clamp(42, 0, 100) == 42
assert isinstance(clamp(42, 0, 100), int)

---

## Solution 4 — Describe a value in words

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** isinstance, strings, dispatch

**Explanation and reasoning**

The ordering is the design: None and bool are tested before the general cases so they are described precisely rather than lumped in with numbers. Mapping is checked before Sequence because a dict is not a Sequence, and the final fallback keeps the function total for any object.

In [ ]:
from collections.abc import Mapping, Sequence


def describe(value) -> str:
    if value is None:
        return 'missing'
    if isinstance(value, bool):
        return f'flag: {value}'
    if isinstance(value, (int, float)):
        return f'number: {value}'
    if isinstance(value, str):
        return f'text: {value}'
    if isinstance(value, Mapping):
        return f'mapping of {len(value)}'
    if isinstance(value, Sequence):
        return f'sequence of {len(value)}'
    return type(value).__name__

**Complexity**

Time O(1) apart from formatting the value; space O(1).

**Edge cases and failure modes**

An empty list yields 'sequence of 0'; bytes yield 'bytes'.

**Alternative approaches**

A match statement on type(value) is faster but cannot cover subclasses.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert describe(48.0) == 'number: 48.0'
assert describe(None) == 'missing'
assert describe(True) == 'flag: True'
assert describe([1, 2, 3]) == 'sequence of 3'
assert describe({'a': 1}) == 'mapping of 1'

---

## Solution 5 — Validate a sensor reading strictly

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** isinstance, bool, validation

**Explanation and reasoning**

Excluding bool explicitly is the whole point of this function: because bool inherits from int, a naive numeric check would accept True for a battery reading. Chained comparison then expresses the inclusive range in one readable expression.

In [ ]:
def is_valid_reading(value, low, high) -> bool:
    if isinstance(value, bool):
        return False
    if not isinstance(value, (int, float)):
        return False
    return low <= value <= high

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

A numeric string returns False because it is not a number at all.

**Alternative approaches**

numbers.Real excludes complex numbers but still admits bool.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert is_valid_reading(48.0, 0, 100) is True
assert is_valid_reading(True, 0, 100) is False
assert is_valid_reading('48', 0, 100) is False
assert is_valid_reading(100, 0, 100) is True

---

## Solution 6 — Parse a telemetry line without losing good channels

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** strings, exceptions, error reporting

**Explanation and reasoning**

Continuing after a failure is the whole point: a single corrupt channel must not discard the valid ones. partition limits the split to the first equals sign, so a value containing '=' survives, and sorting the problems makes the report stable for comparison.

In [ ]:
def parse_readings(text: str) -> tuple[dict, list[str]]:
    readings: dict = {}
    problems: list[str] = []
    for chunk in text.split(';'):
        chunk = chunk.strip()
        if not chunk:
            continue
        name, separator, raw = chunk.partition('=')
        if not separator:
            problems.append(f'{chunk!r}: missing =')
            continue
        name = name.strip()
        try:
            readings[name] = float(raw.strip())
        except ValueError:
            problems.append(f"{name}: {raw.strip()!r} is not a number")
    return readings, sorted(problems)

**Complexity**

Time O(n) in the input length; space O(k) for k channels.

**Edge cases and failure modes**

Empty input returns ({}, []) without raising.

**Alternative approaches**

Raising on the first error loses every other reading.

**Tests — run the cell to verify the reference solution**

In [ ]:
readings, problems = parse_readings('a=1.5;b=bad;c=2')
assert readings == {'a': 1.5, 'c': 2.0}
assert problems == ["b: 'bad' is not a number"]
assert parse_readings('') == ({}, [])

---

## Solution 7 — Normalise units to metres

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** dicts, arithmetic, validation

**Explanation and reasoning**

A lookup table replaces a chain of conditionals with a single dictionary access, so adding a unit is a one-line change. The bool guard prevents True becoming 1.0 metres, and `from None` suppresses the KeyError traceback so the caller sees only the clear message.

In [ ]:
FACTORS = {
    'mm': 0.001,
    'cm': 0.01,
    'm': 1.0,
    'km': 1000.0,
    'in': 0.0254,
    'ft': 0.3048,
}


def to_metres(value, unit: str) -> float:
    if not isinstance(value, (int, float)) or isinstance(value, bool):
        raise ValueError(f'not a number: {value!r}')
    try:
        factor = FACTORS[unit]
    except KeyError:
        raise ValueError(f'unknown unit: {unit!r}') from None
    return float(value) * factor

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

An empty string unit raises ValueError; 'm' returns the value unchanged as a float.

**Alternative approaches**

The pint library handles arbitrary units but is a heavy dependency here.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert to_metres(250, 'cm') == 2.5
assert to_metres(1, 'in') == 0.0254
assert to_metres(3, 'm') == 3.0

---

## Solution 8 — Summarise the types in a batch

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** collections.Counter, dicts, type()

**Explanation and reasoning**

Counter tallies the type names in a single pass and keeps bool and int distinct because their type names differ. Building the dict from sorted keys makes the output deterministic, which matters when it is compared in a test or serialised.

In [ ]:
from collections import Counter


def summarise_types(values) -> dict:
    counts = Counter(type(value).__name__ for value in values)
    summary = {name: counts[name] for name in sorted(counts)}
    summary['total'] = sum(counts.values())
    return summary

**Complexity**

Time O(n log k); space O(k) for k distinct types.

**Edge cases and failure modes**

An empty sequence yields {'total': 0}.

**Alternative approaches**

A plain dict with incrementing counts avoids the import.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert summarise_types([1, 2.0, 'x']) == {'float': 1, 'int': 1, 'str': 1, 'total': 3}
assert summarise_types([]) == {'total': 0}
assert summarise_types([True, 1]) == {'bool': 1, 'int': 1, 'total': 2}

---

## Solution 9 — Compare floats with a tolerance

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** floats, abs, math.isclose

**Explanation and reasoning**

math.isclose with rel_tol set to zero makes the comparison purely absolute, so the tolerance means the same thing at 0.001 and at 10,000. Rejecting bool explicitly prevents True from comparing equal to 1.

In [ ]:
import math


def close_enough(a, b, tolerance: float = 1e-9) -> bool:
    for value in (a, b):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f'not a real number: {value!r}')
    return math.isclose(a, b, rel_tol=0.0, abs_tol=tolerance)

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

Exactly equal values return True regardless of the tolerance.

**Alternative approaches**

abs(a - b) <= tolerance is equivalent and needs no import.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert close_enough(0.1 + 0.2, 0.3) is True
assert close_enough(1.0, 1.1) is False
assert close_enough(1.0, 1.0000000001, 1e-9) is True

---

## Solution 10 — Coerce a value through an ordered list of types

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** try/except, callables, error reporting

**Explanation and reasoning**

Iterating in the caller's order is what makes the ladder configurable: int before float turns '48' into an int rather than 48.0. The final raise chains the last underlying exception with `from`, so the original failure is still visible in the traceback.

In [ ]:
def coerce(value, kinds):
    attempts: list[str] = []
    last: Exception | None = None
    for kind in kinds:
        attempts.append(getattr(kind, '__name__', repr(kind)))
        try:
            return kind(value)
        except (TypeError, ValueError) as exc:
            last = exc
    raise ValueError(
        f'could not convert {value!r} using: {", ".join(attempts)}'
    ) from last

**Complexity**

Time O(k * n) for k attempted conversions; space O(k).

**Edge cases and failure modes**

An empty kinds list raises ValueError naming no attempts.

**Alternative approaches**

A lookup of expected input types chooses one converter but cannot fall back.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert coerce('48.5', [int, float, str]) == 48.5
assert coerce('48', [int, float]) == 48
assert coerce('48', [int]) == 48

---

# Robotics challenge solution — ROBO-X Challenge: Telemetry Frame Gate

**Explanation**

Rendering and then re-parsing exercises the conversion path that hardware would normally provide. The injected ERR value becomes a reported problem rather than a raised exception, and the range filter applies the physical limits that the simulator's own clamping would otherwise hide.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'shared').is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root))

from shared.robo_x_sim import SimulatedRobot, SensorError  # noqa: E402

RANGES = {'distance': (0.0, 30.0), 'temperature': (-40.0, 125.0)}


def parse_readings(text):
    """Parse 'name=value' pairs into floats, reporting bad ones."""
    readings, problems = {}, []
    for chunk in text.split(';'):
        chunk = chunk.strip()
        if not chunk:
            continue
        name, separator, raw = chunk.partition('=')
        if not separator:
            problems.append(f'{chunk!r}: missing =')
            continue
        name = name.strip()
        try:
            readings[name] = float(raw.strip())
        except ValueError:
            problems.append(f"{name}: {raw.strip()!r} is not a number")
    return readings, sorted(problems)


def wire_frame(robot, channels) -> str:
    """Render live readings as the semicolon text a real bus would send."""
    parts = []
    for channel in channels:
        try:
            parts.append(f'{channel}={robot.read_sensor(channel)}')
        except SensorError:
            parts.append(f'{channel}=ERR')
    return ';'.join(parts)


def read_frame(robot, channels) -> dict:
    """Parse the wire frame back, validating each reading's range."""
    readings, problems = parse_readings(wire_frame(robot, channels))
    good = {k: v for k, v in readings.items()
            if k not in RANGES or RANGES[k][0] <= v <= RANGES[k][1]}
    return {'readings': good, 'problems': problems, 'count': len(good)}

**Complexity**

Time O(c); space O(c).

**Notes and trade-offs**

A failed sensor appears in problems and is absent from readings.

In [ ]:
robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
frame = read_frame(robot, ['distance', 'temperature'])
assert frame['count'] == 2
assert frame['problems'] == []

## Reflection

For each exercise where your solution differed from the reference, write one sentence explaining *why* yours was better or worse. This is the highest-value part of the notebook:

- A case the reference handles but mine did not.
- A case mine handles better — and what the reference is missing.
- A performance difference that matters at robot scale.